# Function Calling → MCP
### a hands-on companion to the slide deck

This notebook walks through the same story as the animated deck, but every step actually
**runs** — real API calls to Groq, real tool execution, and a tiny working MCP-style
client/server built from scratch so you can see the "N×M → N+M" idea with real code
instead of just a diagram.

**Feynman's rule we're following:** if you can't build a tiny working version of an idea,
you don't understand it yet. So every section ends with running code, not just prose.

Sections:
1. Setup
2. The frozen brain (why a plain LLM call can't answer live questions)
3. Function calling — the full loop, built by hand
4. Multiple tools — how the model *chooses*
5. The N×M problem, quantified
6. Vendor dialects (why one schema doesn't travel)
7. A minimal MCP — client + servers, built from scratch
8. Wiring MCP into the Groq loop — proving N+M
9. The three primitives: Tools, Resources, Prompts
10. Summary


## 1. Setup

Install the Groq SDK and set the API key.

In [ ]:
%pip install -q groq


In [ ]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "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"


In [ ]:
from groq import Groq
import json

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# All models on Groq support tool use; llama-3.3-70b-versatile is a solid,
# fast default. Swap this string if you want to try another hosted model.
MODEL = "llama-3.3-70b-versatile"


## 2. The frozen brain

Before we give the model any tools, let's just *ask* it something that requires
live, real-world information. It was never shown today's weather during training —
so watch what it actually does with the question.


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the exact current temperature in Chennai right now, in Celsius?"}
    ],
)

print(response.choices[0].message.content)


Notice the model either **hedges** ("I don't have real-time access...") or, worse,
**confidently makes something up**. Either way — this is the frozen-brain problem from
the deck. Talking is all it can do. To fix this, it needs a hand.


## 3. Function calling — the full loop, by hand

Recall the 4-step loop from the deck:

1. User asks
2. Model **decides** and emits a structured tool call
3. Your code **executes** the real function
4. The result goes back, and the model writes the final answer

Let's build exactly that. First, a real (mocked) tool, and its schema — the same
JSON-schema anatomy from the deck: `name`, `description`, `parameters`.


In [ ]:
def get_weather(location: str) -> dict:
    """Pretend this hits a real weather API."""
    fake_db = {
        "chennai": {"temp_c": 31, "condition": "sunny"},
        "london":  {"temp_c": 17, "condition": "cloudy"},
        "tokyo":   {"temp_c": 24, "condition": "rainy"},
    }
    return fake_db.get(location.lower(), {"temp_c": 27, "condition": "unknown"})


weather_tool_schema = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, e.g. 'Chennai'"
                }
            },
            "required": ["location"],
        },
    },
}


In [ ]:
# Step 1 + 2: user asks, model decides
messages = [{"role": "user", "content": "What's the weather in Chennai?"}]

first_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[weather_tool_schema],
    tool_choice="auto",
)

msg = first_response.choices[0].message
print("Model wants to call:", msg.tool_calls)


In [ ]:
# Step 3: your code actually executes the function the model asked for
tool_call = msg.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = get_weather(**args)
print("Function executed. Result:", result)


In [ ]:
# Step 4: send the result back so the model can write the final answer
messages.append(msg)  # the model's tool-call turn
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result),
})

final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=[weather_tool_schema],
)

print(final_response.choices[0].message.content)


That's the entire loop from the deck, running for real: **ask → decide → execute
→ answer.** Everything from here builds on top of this same four-step pattern.


## 4. Multiple tools — how the model chooses

Give it several tools at once and see how it picks the right one — this is closer
to a real agent.


In [ ]:
def get_stock_price(ticker: str) -> dict:
    fake_prices = {"AAPL": 231.4, "GOOG": 178.2, "MSFT": 452.9}
    return {"ticker": ticker.upper(), "price": fake_prices.get(ticker.upper(), 100.0)}


def search_notes(query: str) -> dict:
    fake_notes = {
        "mcp": "MCP standardizes how models connect to tools and data sources.",
        "function calling": "Lets a model call real code instead of only generating text.",
    }
    hit = next((v for k, v in fake_notes.items() if k in query.lower()), "No notes found.")
    return {"result": hit}


stock_tool_schema = {
    "type": "function",
    "function": {
        "name": "get_stock_price",
        "description": "Get the current stock price for a ticker symbol.",
        "parameters": {
            "type": "object",
            "properties": {"ticker": {"type": "string", "description": "e.g. 'AAPL'"}},
            "required": ["ticker"],
        },
    },
}

notes_tool_schema = {
    "type": "function",
    "function": {
        "name": "search_notes",
        "description": "Search the user's personal notes for a topic.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
}

all_tools = [weather_tool_schema, stock_tool_schema, notes_tool_schema]
available_functions = {
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "search_notes": search_notes,
}


In [ ]:
def run_agent_loop(user_message: str, tools=all_tools, verbose=True):
    """A tiny, general version of the 4-step loop, reusable for any question."""
    messages = [{"role": "user", "content": user_message}]

    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools, tool_choice="auto"
    )
    msg = response.choices[0].message

    # Model may choose zero, one, or several tools
    if not msg.tool_calls:
        return msg.content

    messages.append(msg)
    for tc in msg.tool_calls:
        fn = available_functions[tc.function.name]
        args = json.loads(tc.function.arguments)
        result = fn(**args)
        if verbose:
            print(f"  -> called {tc.function.name}({args}) = {result}")
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

    final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    return final.choices[0].message.content


for q in [
    "What's the weather in Tokyo?",
    "What's AAPL trading at?",
    "What do my notes say about MCP?",
]:
    print("Q:", q)
    print("A:", run_agent_loop(q))
    print()


## 5. The N×M problem, quantified

In the deck: 3 models × 4 tools = 12 hand-wired integrations. Let's just compute
that directly — the "math" is genuinely this simple, which is exactly why it hurts
at scale.


In [ ]:
models = ["gpt", "claude", "gemini"]
tools_list = ["weather", "search", "calendar", "database"]

integrations = [(m, t) for m in models for t in tools_list]
print(f"{len(models)} models x {len(tools_list)} tools = {len(integrations)} custom integrations")
for m, t in integrations:
    print(f"  {m:8s} <-> {t}")


Every one of those pairs is, in the function-calling world, its own schema, its own glue code, its own edge cases.

## 6. Vendor dialects

The *concept* of a tool is identical everywhere. The *shape* of the JSON is not.
Here's the same `get_weather` tool, described three ways — no API calls needed,
just look at the structural differences.


In [ ]:
openai_style = {
    "type": "function",
    "function": {"name": "get_weather", "parameters": {"location": "string"}},
}

anthropic_style = {
    "name": "get_weather",
    "input_schema": {"type": "object", "properties": {"location": {"type": "string"}}},
}

gemini_style = {
    "function_declarations": [
        {"name": "get_weather", "parameters": {"location": "string"}}
    ]
}

for name, schema in [("OpenAI", openai_style), ("Anthropic", anthropic_style), ("Gemini", gemini_style)]:
    print(f"--- {name} ---")
    print(json.dumps(schema, indent=2))
    print()


Same idea, three incompatible shapes. This is the maintenance tax MCP is designed to remove.

## 7. A minimal MCP — built from scratch

We won't pull in a full MCP SDK here — instead we'll build the *smallest possible*
version of the idea, so the mechanism is obvious:

- A **server** exposes `list_tools()` and `call_tool(name, args)` — nothing model-specific.
- A **client** talks to *any* server through that same two-method interface.
- The model never talks to servers directly — only the client does, using schemas
  the client itself generated from `list_tools()`.

This mirrors the Host → Client → Server architecture from the deck.


In [ ]:
class MCPServer:
    """A minimal illustrative MCP-style server."""
    def __init__(self, name):
        self.name = name
        self._tools = {}     # name -> (fn, schema)
        self._resources = {} # name -> value
        self._prompts = {}   # name -> template string

    def tool(self, name, description, parameters):
        def decorator(fn):
            self._tools[name] = {
                "fn": fn,
                "schema": {
                    "type": "function",
                    "function": {"name": name, "description": description, "parameters": parameters},
                },
            }
            return fn
        return decorator

    def resource(self, name, value):
        self._resources[name] = value

    def prompt(self, name, template):
        self._prompts[name] = template

    def list_tools(self):
        return [t["schema"] for t in self._tools.values()]

    def call_tool(self, name, args):
        return self._tools[name]["fn"](**args)

    def list_resources(self):
        return list(self._resources.keys())

    def read_resource(self, name):
        return self._resources[name]

    def list_prompts(self):
        return list(self._prompts.keys())

    def get_prompt(self, name, **kwargs):
        return self._prompts[name].format(**kwargs)


In [ ]:
# --- Weather server ---
weather_server = MCPServer("weather-server")

@weather_server.tool(
    "get_weather",
    "Get current weather for a city",
    {"type": "object", "properties": {"location": {"type": "string"}}, "required": ["location"]},
)
def _get_weather(location):
    return get_weather(location)

weather_server.resource("last_updated", "2026-07-26T09:00:00Z")
weather_server.prompt("weather_report", "Give a short, friendly weather report for {city}.")


# --- Database server ---
db_server = MCPServer("database-server")

@db_server.tool(
    "get_stock_price",
    "Get current stock price for a ticker",
    {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]},
)
def _get_stock_price(ticker):
    return get_stock_price(ticker)

db_server.resource("schema_version", "3.2")


In [ ]:
class MCPClient:
    """A minimal client: talks to any number of servers through one interface."""
    def __init__(self, servers):
        self.servers = servers            # list of MCPServer
        self._tool_owner = {}              # tool name -> server

    def discover_tools(self):
        all_schemas = []
        for server in self.servers:
            for schema in server.list_tools():
                name = schema["function"]["name"]
                self._tool_owner[name] = server
                all_schemas.append(schema)
        return all_schemas

    def call(self, name, args):
        return self._tool_owner[name].call_tool(name, args)


client_mcp = MCPClient(servers=[weather_server, db_server])
discovered = client_mcp.discover_tools()
print(json.dumps(discovered, indent=2))


Notice what just happened: the client discovered the tools **at runtime**, by asking
the servers what they offer — nothing was hardcoded. This is the "discovery" property
from the comparison table that plain function calling doesn't have.


## 8. Wiring MCP into the Groq loop — proving N+M

Now let's plug the *same* `run_agent_loop` pattern from Section 4 into the MCP client
instead of a hardcoded `available_functions` dict. Adding a brand-new tool now means
writing **one server**, not touching the model-facing code at all.


In [ ]:
def run_agent_loop_via_mcp(user_message: str, mcp_client: MCPClient, verbose=True):
    tools = mcp_client.discover_tools()   # <-- generated, not hand-maintained
    messages = [{"role": "user", "content": user_message}]

    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools, tool_choice="auto"
    )
    msg = response.choices[0].message
    if not msg.tool_calls:
        return msg.content

    messages.append(msg)
    for tc in msg.tool_calls:
        args = json.loads(tc.function.arguments)
        result = mcp_client.call(tc.function.name, args)   # <-- routed generically
        if verbose:
            print(f"  -> [{tc.function.name}] via MCP = {result}")
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

    final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    return final.choices[0].message.content


print(run_agent_loop_via_mcp("What's the weather in London?", client_mcp))
print(run_agent_loop_via_mcp("What's MSFT worth right now?", client_mcp))


In [ ]:
# Add a THIRD server with zero changes to run_agent_loop_via_mcp or the model-facing code.
notes_server = MCPServer("notes-server")

@notes_server.tool(
    "search_notes",
    "Search the user's personal notes for a topic",
    {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
)
def _search_notes(query):
    return search_notes(query)

client_mcp.servers.append(notes_server)

print(run_agent_loop_via_mcp("What do my notes say about function calling?", client_mcp))


That's the whole point of the deck's math slide, made concrete: **3 servers, 1
client, 0 lines changed in the agent loop.** Compare that to Section 5, where every
new tool meant a new `if/elif` branch and a new schema hand-copied into every model's
dialect.


## 9. The three primitives: Tools, Resources, Prompts

Function calling only ever gave you Tools. Our tiny `MCPServer` also exposes
**Resources** (readable data, not actions) and **Prompts** (reusable templates) —
the other two primitives from the deck.


In [ ]:
print("Resources on weather-server:", weather_server.list_resources())
print("Reading one:", weather_server.read_resource("last_updated"))
print()
print("Prompts on weather-server:", weather_server.list_prompts())
print("Rendered prompt:", weather_server.get_prompt("weather_report", city="Chennai"))


In [ ]:
# A resource or prompt can feed straight into a normal chat call - no tool-calling needed
rendered_prompt = weather_server.get_prompt("weather_report", city="Chennai")

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": rendered_prompt}],
)
print(resp.choices[0].message.content)


## 10. Summary

| | Function calling | MCP (this notebook's mini version) |
|---|---|---|
| Standard | one per vendor | one shared interface (`list_tools` / `call_tool`) |
| Discovery | hardcoded `available_functions` dict | `discover_tools()` at runtime |
| Adding a tool | new schema + new branch, per model | one new `MCPServer` |
| What's exposed | actions only | actions (Tools) + data (Resources) + templates (Prompts) |
| Growth | N models × M tools | N models + M servers |

**Function calling gave a model a hand. MCP gave every hand a universal socket.**

Next steps you could try:
- Swap the mock functions for real APIs (a real weather API, a real DB).
- Swap `MODEL` for another Groq-hosted model — tool use works the same way on all of them.
- Replace the in-notebook `MCPServer`/`MCPClient` toy classes with the official `mcp` Python SDK once you're ready to run a real, separate server process.
